# 03 — Évaluation du RAG simple (référence)

Ce notebook mesure la qualité du RAG simple sur le jeu de questions **dev**. Ces résultats servent de référence (*baseline*) pour le système multi-agents.

Le jeu **test** n'est pas utilisé ici : il est réservé à la comparaison finale, pour que les réglages ne soient jamais faits sur les questions qui servent à juger le système.

Les réponses ont été générées avec `scripts/evaluate.py --split dev`, exécuté sur GPU dans un notebook Kaggle (`kaggle/evaluation_kaggle.ipynb`).

In [ ]:
import json

import pandas as pd

from assistant_rag.config import RESULTS_DIR
from assistant_rag.evaluation.dataset import load_questions
from assistant_rag.evaluation.metrics import score_records, summarize
from assistant_rag.evaluation.runner import load_records

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.precision", 2)

run_dir = RESULTS_DIR / "simple-dev"
config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
records = load_records(run_dir / "answers.jsonl")
scores = score_records(records).set_index("id")
answers = pd.DataFrame(records).set_index("id")

## 1. Jeu de questions

60 questions écrites à partir du texte des deux règlements, chacune avec une réponse de référence et les articles qui la justifient :

- **factuelles** : la réponse se trouve dans un seul article ;
- **multi-articles** : la réponse combine deux articles, parfois des deux règlements ;
- **hors corpus** : la réponse n'est pas dans les documents ; le système doit refuser de répondre.

In [ ]:
questions = pd.DataFrame([q.__dict__ for q in load_questions()])
pd.crosstab(questions["type"], questions["split"], margins=True, margins_name="total")

## 2. Configuration évaluée

In [ ]:
settings = config["settings"]
pd.Series(
    {
        "LLM": settings["llm_model"],
        "Embeddings": settings["embed_model"],
        "Taille des chunks": settings["chunk_size"],
        "Passages retrouvés (k)": settings["top_k"],
        "Version du prompt": config["prompt_version"],
        "LLM-juge": config["judge"],
        "Commit": config["git_commit"],
    },
    name="valeur",
).to_frame()

## 3. Résultats

| Métrique | Définition |
| --- | --- |
| Hit@k | part des questions dont au moins un article attendu figure parmi les k passages retrouvés |
| Rappel@k | part des articles attendus retrouvés |
| MRR | moyenne de 1 / rang du premier article attendu retrouvé |
| Précision des citations | part des passages cités qui appartiennent à un article attendu |
| Rappel des citations | part des articles attendus effectivement cités |
| Exactitude | note du LLM-juge : 1 si correct, 0,5 si partiel, 0 si incorrect ou refus |
| Refus à tort | part des questions avec réponse pour lesquelles le système a refusé de répondre |
| Refus correct | part des questions hors corpus pour lesquelles le système a refusé de répondre |

La précision des citations est stricte : citer un article pertinent mais non listé dans les articles attendus compte comme une erreur.

In [ ]:
summarize(records)

## 4. Exactitude selon le LLM-juge

In [ ]:
judged = answers[answers["judge"].notna()]
verdicts = judged["judge"].apply(lambda j: j["verdict"]).rename("verdict")
verdicts.value_counts().rename("questions").to_frame()

## 5. Analyse des erreurs

### 5.1 Recherche : article attendu absent des passages retrouvés

In [ ]:
def articles(items):
    return ", ".join(f"{i['document']} {i['article']}" for i in items)


misses = scores[scores["hit"] == 0].index
answers.loc[misses].assign(
    attendus=lambda d: d["expected"].apply(lambda e: ", ".join(f"{doc} {art}" for doc, art in e)),
    retrouvés=lambda d: d["retrieved"].apply(articles),
)[["type", "question", "attendus", "retrouvés"]]

### 5.2 Réponses jugées partielles ou incorrectes

In [ ]:
wrong = verdicts[verdicts != "correct"].index
answers.loc[wrong].assign(
    verdict=lambda d: d["judge"].apply(lambda j: j["verdict"]),
    justification=lambda d: d["judge"].apply(lambda j: j["justification"]),
    recherche_ok=scores.loc[wrong, "hit"].astype(bool),
)[["question", "verdict", "recherche_ok", "justification"]]

La colonne `recherche_ok` sépare les deux origines possibles d'une erreur : si le bon article a été retrouvé, l'erreur vient de la génération ; sinon, elle vient de la recherche.

### 5.3 Refus

In [ ]:
refusals = scores[(scores["false_refusal"] == 1) | (scores["correct_refusal"] == 0)].index
answers.loc[refusals, ["type", "question", "refused", "answer"]]

### 5.4 Citations hors des articles attendus

In [ ]:
imprecise = scores[scores["citation_precision"] < 1].index
answers.loc[imprecise].assign(
    attendus=lambda d: d["expected"].apply(lambda e: ", ".join(f"{doc} {art}" for doc, art in e)),
    cités=lambda d: d["cited"].apply(articles),
)[["question", "attendus", "cités"]]

## 6. Contrôle manuel du LLM-juge

Le LLM-juge peut se tromper. Ses verdicts sont relus sur un échantillon de réponses, en comparant chaque réponse à la référence.

In [ ]:
sample = judged.sample(n=min(10, len(judged)), random_state=0)
for qid, row in sample.iterrows():
    print(f"{qid} — verdict du juge : {row['judge']['verdict']}")
    print(f"Question  : {row['question']}")
    print(f"Référence : {row['reference']}")
    print(f"Réponse   : {row['answer']}")
    print("-" * 100)

## 7. Temps de réponse sur GPU

In [ ]:
scores["latency"].describe(percentiles=[0.5, 0.9]).rename("secondes").to_frame()